# DeepResearch Task 4: Context-Aware Next-Query Generation

This notebook is the primary review and reproduction entry point for Task 4.

The model receives:

1. a research question;
2. previously generated search queries;
3. descriptions of already visited sources;

and generates **one next query for downstream search**.

> Scope clarification: this repository generates a search query. It does not execute a web search or retrieve web pages.

## End-to-end workflow

```text
Task 3 source descriptions + Task 4 trajectories
                    ↓
          real SFT train/validation
                    ↓
     optional filtered synthetic augmentation
                    ↓
            LoRA fine-tuning
                    ↓
         deterministic inference
                    ↓
 Token F1 / Jaccard / BERTScore / LLM judge
```

The notebook runs in a fast review mode by default and reads committed datasets, predictions, and evaluation artifacts. Expensive stages can be enabled explicitly below.

In [34]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from collections import Counter
from pathlib import Path
from statistics import mean
from typing import Any

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / "README.md").exists():
    candidates = [ROOT.parent, ROOT / "deepresearch-next-query-generation"]
    for candidate in candidates:
        if (candidate / "README.md").exists():
            ROOT = candidate.resolve()
            break

os.chdir(ROOT)
print("Project root:", ROOT)
print("Python:", sys.executable)

Project root: C:\Users\Dominus\Desktop\deepresearch_summerschool
Python: c:\Users\Dominus\Desktop\deepresearch_summerschool\.venv\Scripts\python.exe


## 1. Execution switches

All expensive operations are disabled by default.

- `RUN_SYNTHETIC_GENERATION`: calls an external teacher model and requires API credentials.
- `RUN_TRAINING`: fine-tunes the mixed-data LoRA adapter.
- `RUN_INFERENCE`: generates base, real-only, and mixed-model predictions.
- `RUN_BERTSCORE`: computes BERTScore for the mixed run.
- `RUN_LLM_JUDGE`: performs paid external LLM-as-a-judge calls.

Set only the required flags to `True`.

In [35]:
RUN_SYNTHETIC_GENERATION = False
RUN_TRAINING = False
RUN_INFERENCE = False
RUN_BERTSCORE = False
RUN_LLM_JUDGE = False

TARGET_SYNTHETIC_EXAMPLES = 100
MAX_SOURCE_EXAMPLES = 180
LLM_JUDGE_REPEATS = 2

PYTHON = sys.executable

In [36]:
def run_command(
    args: list[str],
    *,
    extra_env: dict[str, str] | None = None,
) -> None:
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)

    printable = " ".join(args)
    print(f"$ {printable}")
    subprocess.run(args, cwd=ROOT, env=env, check=True)


PATHS = {
    "real_train": ROOT / "data/lora/train_real_sft.jsonl",
    "mixed_train": ROOT / "data/lora/train_mixed_pilot_sft.jsonl",
    "real_validation": ROOT / "data/lora/val_real_sft.jsonl",
    "synthetic": ROOT / "data/synthetic/task4_synthetic_pilot.jsonl",
    "synthetic_audit": ROOT / "results/synthetic_dataset_audit.json",
    "synthetic_summary": ROOT / "results/synthetic_pilot_summary.json",
    "mixed_training_metrics": ROOT / "results/qwen2.5_1.5b_lora_mixed_metrics.json",
    "real_training_metrics": ROOT / "results/qwen2.5_1.5b_lora_real_metrics.json",
    "llm_judge": ROOT / "results/llm_judge_real_vs_mixed_summary.json",
}

PREDICTION_PATHS = {
    "Base Qwen 1.5B": ROOT / "runs/qwen2.5_1.5b_base_predictions.jsonl",
    "Real-only LoRA": ROOT / "runs/qwen2.5_1.5b_lora_real_predictions.jsonl",
    "Mixed LoRA": ROOT / "runs/qwen2.5_1.5b_lora_mixed_predictions.jsonl",
}

CONFIG_PATHS = {
    "training_mixed": ROOT / "configs/qwen2.5_1.5b_mixed.json",
    "inference_base": ROOT / "configs/qwen2.5_1.5b_base_inference.json",
    "inference_real": ROOT / "configs/qwen2.5_1.5b_real_inference.json",
    "inference_mixed": ROOT / "configs/qwen2.5_1.5b_mixed_inference.json",
}

required_files = {
    **PATHS,
    **PREDICTION_PATHS,
    **CONFIG_PATHS,
}

status = pd.DataFrame(
    [
        {
            "artifact": name,
            "path": str(path.relative_to(ROOT)) if path.is_absolute() else str(path),
            "exists": path.exists(),
        }
        for name, path in required_files.items()
    ]
)
display(status)

,artifact,path,exists
0,real_train,data\lora\train_real_sft.jsonl,True
1,mixed_train,data\lora\train_mixed_pilot_sft.jsonl,True
2,real_validation,data\lora\val_real_sft.jsonl,True
3,synthetic,data\synthetic\task4_synthetic_pilot.jsonl,True
4,synthetic_audit,results\synthetic_dataset_audit.json,True
5,synthetic_summary,results\synthetic_pilot_summary.json,True
6,mixed_training_metrics,results\qwen2.5_1.5b_lora_mixed_metrics.json,True
7,real_training_metrics,results\qwen2.5_1.5b_lora_real_metrics.json,True
8,llm_judge,results\llm_judge_real_vs_mixed_summary.json,True
9,Base Qwen 1.5B,runs\qwen2.5_1.5b_base_predictions.jsonl,True


## 2. Dataset inspection and split validation

In [37]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as file:
        value = json.load(file)
    if not isinstance(value, dict):
        raise ValueError(f"Expected JSON object: {path}")
    return value


def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {path}, line {line_number}: {error}"
                ) from error
            if not isinstance(row, dict):
                raise ValueError(
                    f"Expected JSON object in {path}, line {line_number}"
                )
            rows.append(row)
    return rows


def extract_research_question(item: dict[str, Any]) -> str | None:
    for key in ("research_question", "question"):
        value = item.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    metadata = item.get("metadata")
    if isinstance(metadata, dict):
        for key in ("research_question", "question"):
            value = metadata.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

    messages = item.get("messages")
    if isinstance(messages, list):
        user_text = "\n".join(
            str(message.get("content", ""))
            for message in messages
            if message.get("role") == "user"
        )
        patterns = (
            r"Research question:\s*(.+?)(?:\n\s*\n|\nPrevious queries:|\nVisited sources:)",
            r"Исследовательский вопрос:\s*(.+?)(?:\n\s*\n|\nПредыдущие запросы:|\nПосещённые источники:)",
        )
        for pattern in patterns:
            match = re.search(pattern, user_text, flags=re.IGNORECASE | re.DOTALL)
            if match:
                return " ".join(match.group(1).split())

    return None


dataset_rows = []
dataset_cache: dict[str, list[dict[str, Any]]] = {}

for name in ("real_train", "mixed_train", "real_validation", "synthetic"):
    path = PATHS[name]
    if not path.exists():
        continue

    rows = load_jsonl(path)
    dataset_cache[name] = rows
    questions = {
        question
        for item in rows
        if (question := extract_research_question(item))
    }

    dataset_rows.append(
        {
            "dataset": name,
            "examples": len(rows),
            "parsed_unique_research_questions": len(questions),
            "path": str(path.relative_to(ROOT)),
        }
    )

dataset_table = pd.DataFrame(dataset_rows)
display(dataset_table)

train_questions = {
    question
    for item in dataset_cache.get("real_train", [])
    if (question := extract_research_question(item))
}
validation_questions = {
    question
    for item in dataset_cache.get("real_validation", [])
    if (question := extract_research_question(item))
}

if train_questions and validation_questions:
    overlap = train_questions & validation_questions
    print("Parsed train research questions:", len(train_questions))
    print("Parsed validation research questions:", len(validation_questions))
    print("Train/validation research-question overlap:", len(overlap))
    assert not overlap, "Research-question leakage detected."
else:
    print(
        "Research-question leakage check was not executed because the notebook "
        "could not reliably parse question metadata from one of the SFT files."
    )

,dataset,examples,parsed_unique_research_questions,path
0,real_train,248,14,data\lora\train_real_sft.jsonl
1,mixed_train,348,14,data\lora\train_mixed_pilot_sft.jsonl
2,real_validation,32,3,data\lora\val_real_sft.jsonl
3,synthetic,100,13,data\synthetic\task4_synthetic_pilot.jsonl


Parsed train research questions: 14
Parsed validation research questions: 3
Train/validation research-question overlap: 0


## 3. Optional synthetic-data generation and audit

In [38]:
if RUN_SYNTHETIC_GENERATION:
    run_command(
        [PYTHON, "generate_synthetic_train_pilot.py"],
        extra_env={
            "TARGET_SYNTHETIC_EXAMPLES": str(TARGET_SYNTHETIC_EXAMPLES),
            "MAX_SOURCE_EXAMPLES": str(MAX_SOURCE_EXAMPLES),
        },
    )
    run_command([PYTHON, "clean_synthetic_queries.py"])
    run_command([PYTHON, "analyze_synthetic_dataset.py"])
else:
    print("Synthetic generation skipped. Reading committed audit artifacts.")

Synthetic generation skipped. Reading committed audit artifacts.


In [39]:
def flatten_dict(
    value: dict[str, Any],
    prefix: str = "",
) -> dict[str, Any]:
    flattened: dict[str, Any] = {}
    for key, child in value.items():
        full_key = f"{prefix}.{key}" if prefix else key
        if isinstance(child, dict):
            flattened.update(flatten_dict(child, full_key))
        elif not isinstance(child, (list, dict)):
            flattened[full_key] = child
    return flattened


audit_records = []
for label, path in (
    ("generation_summary", PATHS["synthetic_summary"]),
    ("dataset_audit", PATHS["synthetic_audit"]),
):
    if not path.exists():
        continue
    flattened = flatten_dict(load_json(path))
    for key, value in flattened.items():
        audit_records.append(
            {
                "artifact": label,
                "metric": key,
                "value": value,
            }
        )

audit_table = pd.DataFrame(audit_records)
if audit_table.empty:
    print("No synthetic audit artifacts found.")
else:
    selected_patterns = (
        "accepted",
        "real_train",
        "mixed_train",
        "synthetic_share",
        "duplicate",
        "overlap",
        "outside",
        "mean",
        "median",
        "language",
        "unsupported_numbers",
    )
    selected = audit_table[
        audit_table["metric"].str.lower().apply(
            lambda name: any(pattern in name for pattern in selected_patterns)
        )
    ]
    display(selected.reset_index(drop=True))

,artifact,metric,value
0,generation_summary,real_train_examples,248
1,generation_summary,accepted_synthetic_examples,10
2,generation_summary,mixed_train_examples,258
3,generation_summary,language_policy,same_as_parent_real_target
4,generation_summary,accepted_language_distribution.English,8
5,generation_summary,accepted_language_distribution.Russian,2
6,generation_summary,rejection_reasons.unsupported_numbers,2
7,dataset_audit,mixed_train_examples,348
8,dataset_audit,synthetic_share,0.287356
9,dataset_audit,query_token_lengths.mean,9.71


## 4. Optional LoRA training

In [40]:
if RUN_TRAINING:
    run_command(
        [
            PYTHON,
            "train_lora_pipeline.py",
            "--config",
            str(CONFIG_PATHS["training_mixed"].relative_to(ROOT)),
        ]
    )
else:
    print("Training skipped. Existing adapter and metrics are used.")

Training skipped. Existing adapter and metrics are used.


In [41]:
training_rows = []

for model_name, path in (
    ("Real-only LoRA", PATHS["real_training_metrics"]),
    ("Mixed LoRA", PATHS["mixed_training_metrics"]),
):
    if not path.exists():
        continue

    metrics = load_json(path)
    flat = flatten_dict(metrics)

    def first_matching(*suffixes: str) -> Any:
        for suffix in suffixes:
            for key, value in flat.items():
                if key.endswith(suffix):
                    return value
        return None

    training_rows.append(
        {
            "model": model_name,
            "training_examples": first_matching(
                "training_examples",
                "dataset_stats.train.examples",
            ),
            "validation_examples": first_matching(
                "validation_examples",
                "dataset_stats.validation.examples",
            ),
            "best_eval_loss": first_matching(
                "best_metric",
                "eval_metrics.eval_loss",
                "final_eval_metrics.eval_loss",
            ),
            "runtime_seconds": first_matching(
                "runtime_seconds",
                "train_metrics.train_runtime",
            ),
            "peak_vram_gb": first_matching("peak_vram_gb"),
            "best_checkpoint": first_matching("best_checkpoint"),
        }
    )

display(pd.DataFrame(training_rows))

,model,training_examples,validation_examples,best_eval_loss,runtime_seconds,peak_vram_gb,best_checkpoint
0,Real-only LoRA,248,32,3.194903,404.30240,NaN,NaN
1,Mixed LoRA,348,32,3.316849,504.03698,6.845305,models\qwen2.5-1.5b-task4-lora-mixed\checkpoin...


## 5. Optional deterministic inference

In [42]:
if RUN_INFERENCE:
    for config_name in ("inference_base", "inference_real", "inference_mixed"):
        run_command(
            [
                PYTHON,
                "generate_predictions_pipeline.py",
                "--config",
                str(CONFIG_PATHS[config_name].relative_to(ROOT)),
            ]
        )
else:
    print("Inference skipped. Reading committed prediction files.")

Inference skipped. Reading committed prediction files.


## 6. Lexical evaluation

In [43]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9%°\-]+")


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def token_f1(prediction: str, target: str) -> float:
    pred_tokens = tokenize(prediction)
    target_tokens = tokenize(target)

    if not pred_tokens or not target_tokens:
        return 0.0

    pred_counts = Counter(pred_tokens)
    target_counts = Counter(target_tokens)
    overlap = sum(
        min(count, target_counts.get(token, 0))
        for token, count in pred_counts.items()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(target_tokens)
    return 2 * precision * recall / (precision + recall)


def jaccard(prediction: str, target: str) -> float:
    pred_set = set(tokenize(prediction))
    target_set = set(tokenize(target))
    if not pred_set or not target_set:
        return 0.0
    return len(pred_set & target_set) / len(pred_set | target_set)


def get_text(item: dict[str, Any], keys: tuple[str, ...]) -> str:
    for key in keys:
        value = item.get(key)
        if isinstance(value, str):
            return value
    raise KeyError(f"None of the keys {keys} found in prediction row.")


def lexical_metrics(path: Path) -> dict[str, Any]:
    rows = load_jsonl(path)
    f1_scores = []
    jaccard_scores = []
    exact_scores = []
    repeat_scores = []
    pred_lengths = []
    target_lengths = []

    for item in rows:
        prediction = get_text(
            item,
            ("generated_query", "prediction", "pred", "output", "generated"),
        )
        target = get_text(
            item,
            ("target_query", "target", "reference", "gold", "expected"),
        )
        previous_queries = item.get("previous_queries", [])

        f1_scores.append(token_f1(prediction, target))
        jaccard_scores.append(jaccard(prediction, target))
        exact_scores.append(
            float(prediction.strip().lower() == target.strip().lower())
        )
        repeat_scores.append(
            float(
                any(
                    prediction.strip().lower() == str(query).strip().lower()
                    for query in previous_queries
                )
            )
        )
        pred_lengths.append(len(tokenize(prediction)))
        target_lengths.append(len(tokenize(target)))

    return {
        "examples": len(rows),
        "exact_match": mean(exact_scores),
        "token_f1": mean(f1_scores),
        "jaccard": mean(jaccard_scores),
        "repeated_previous_query": mean(repeat_scores),
        "average_query_length": mean(pred_lengths),
        "average_target_length": mean(target_lengths),
    }


lexical_rows = []
for model_name, path in PREDICTION_PATHS.items():
    if not path.exists():
        continue
    lexical_rows.append(
        {
            "model": model_name,
            **lexical_metrics(path),
        }
    )

lexical_table = pd.DataFrame(lexical_rows)
display(lexical_table.round(4))

,model,examples,exact_match,token_f1,jaccard,repeated_previous_query,average_query_length,average_target_length
0,Base Qwen 1.5B,32,0.0,0.2092,0.1256,0.0,17.0312,10.9375
1,Real-only LoRA,32,0.0,0.2199,0.1346,0.0,11.4062,10.9375
2,Mixed LoRA,32,0.0,0.2296,0.1422,0.0,13.0312,10.9375


## 7. BERTScore

In [44]:
if RUN_BERTSCORE:
    run_command(
        [
            PYTHON,
            "evaluate_bertscore_single.py",
            "--predictions",
            "runs/qwen2.5_1.5b_lora_mixed_predictions.jsonl",
            "--output",
            "results/qwen2.5_1.5b_lora_mixed_bertscore.json",
        ]
    )
else:
    print("BERTScore recomputation skipped. Reading committed artifacts.")

BERTScore recomputation skipped. Reading committed artifacts.


In [45]:
BERTSCORE_PATHS = {
    "Base Qwen 1.5B": ROOT / "results/bertscore_qwen2.5_1.5b_base.csv",
    "Real-only LoRA": ROOT / "results/bertscore_qwen2.5_1.5b_lora_real.csv",
    "Mixed LoRA": ROOT / "results/qwen2.5_1.5b_lora_mixed_bertscore.json",
}


def load_bertscore(path: Path) -> dict[str, float] | None:
    if not path.exists():
        return None

    if path.suffix.lower() == ".json":
        data = load_json(path)
        return {
            "bertscore_precision": float(data["bertscore_precision"]),
            "bertscore_recall": float(data["bertscore_recall"]),
            "bertscore_f1": float(data["bertscore_f1"]),
        }

    frame = pd.read_csv(path)
    lower_columns = {column.lower(): column for column in frame.columns}

    def column_mean(candidates: tuple[str, ...]) -> float | None:
        for candidate in candidates:
            if candidate in lower_columns:
                return float(frame[lower_columns[candidate]].mean())
        for lower_name, original_name in lower_columns.items():
            if any(candidate in lower_name for candidate in candidates):
                return float(frame[original_name].mean())
        return None

    precision = column_mean(("bertscore_precision", "precision", "p"))
    recall = column_mean(("bertscore_recall", "recall", "r"))
    f1 = column_mean(("bertscore_f1", "f1"))

    if precision is None or recall is None or f1 is None:
        return None

    return {
        "bertscore_precision": precision,
        "bertscore_recall": recall,
        "bertscore_f1": f1,
    }


bertscore_rows = []
for model_name, path in BERTSCORE_PATHS.items():
    score_values = load_bertscore(path)
    if score_values is not None:
        bertscore_rows.append({"model": model_name, **score_values})

bertscore_table = pd.DataFrame(bertscore_rows)
display(bertscore_table.round(4))

,model,bertscore_precision,bertscore_recall,bertscore_f1
0,Base Qwen 1.5B,0.8396,0.8473,0.8433
1,Real-only LoRA,0.8495,0.8553,0.8522
2,Mixed LoRA,0.8521,0.8585,0.8552


## 8. Combined automatic metrics

In [46]:
if lexical_table.empty:
    combined_metrics = bertscore_table.copy()
elif bertscore_table.empty:
    combined_metrics = lexical_table.copy()
else:
    combined_metrics = lexical_table.merge(
        bertscore_table,
        on="model",
        how="outer",
    )

preferred_columns = [
    "model",
    "examples",
    "token_f1",
    "jaccard",
    "bertscore_f1",
    "average_query_length",
    "repeated_previous_query",
]
combined_metrics = combined_metrics[
    [column for column in preferred_columns if column in combined_metrics.columns]
]
display(combined_metrics.round(4))

,model,examples,token_f1,jaccard,bertscore_f1,average_query_length,repeated_previous_query
0,Base Qwen 1.5B,32,0.2092,0.1256,0.8433,17.0312,0.0
1,Mixed LoRA,32,0.2296,0.1422,0.8552,13.0312,0.0
2,Real-only LoRA,32,0.2199,0.1346,0.8522,11.4062,0.0


## 9. Blind pairwise LLM-as-a-judge

In [47]:
if RUN_LLM_JUDGE:
    run_command(
        [
            PYTHON,
            "evaluate_llm_judge_pairwise.py",
            "--repeats",
            str(LLM_JUDGE_REPEATS),
        ]
    )
else:
    print("LLM-as-a-judge skipped. Reading committed summary.")

LLM-as-a-judge skipped. Reading committed summary.


In [48]:
judge_path = PATHS["llm_judge"]

if not judge_path.exists():
    print("LLM judge summary not found.")
else:
    judge = load_json(judge_path)

    outcomes = judge.get("per_example_outcomes", {})
    outcome_table = pd.DataFrame(
        [
            {
                "outcome": name,
                "examples": count,
                "share": count / judge.get("judged_examples", 1),
            }
            for name, count in outcomes.items()
        ]
    )
    display(outcome_table.round(4))

    real_scores = judge.get("real_mean_scores", {})
    mixed_scores = judge.get("mixed_mean_scores", {})
    criteria = sorted(set(real_scores) | set(mixed_scores))

    criteria_table = pd.DataFrame(
        [
            {
                "criterion": criterion,
                "real_only": real_scores.get(criterion),
                "mixed": mixed_scores.get(criterion),
                "mixed_minus_real": (
                    mixed_scores.get(criterion) - real_scores.get(criterion)
                    if criterion in real_scores and criterion in mixed_scores
                    else None
                ),
            }
            for criterion in criteria
        ]
    )
    display(criteria_table.round(4))

,outcome,examples,share
0,mixed,12,0.3750
1,real,13,0.4062
2,tie,7,0.2188


,criterion,real_only,mixed,mixed_minus_real
0,groundedness,3.7812,3.3906,-0.3906
1,novelty,2.3594,2.5156,0.1562
2,overall,3.0781,3.1094,0.0312
3,relevance,3.4062,3.3750,-0.0312
4,search_quality,3.5156,3.2812,-0.2344
5,technical_specificity,3.0625,3.3125,0.2500


## 10. Qualitative examples

In [49]:
sample_frames = []

for model_name, path in PREDICTION_PATHS.items():
    if not path.exists():
        continue

    rows = load_jsonl(path)
    for index, item in enumerate(rows[:5], start=1):
        sample_frames.append(
            {
                "example": index,
                "model": model_name,
                "research_question": str(
                    item.get("research_question", "")
                )[:180],
                "target_query": item.get("target_query", item.get("target", "")),
                "generated_query": item.get(
                    "generated_query",
                    item.get("prediction", ""),
                ),
            }
        )

samples = pd.DataFrame(sample_frames)
display(samples)

,example,model,research_question,target_query,generated_query
0,1,Base Qwen 1.5B,What are the distinct mechanisms of refractory...,"Influence of decarburized layer thickness, por...",mechanism + material + process condition + oxy...
1,2,Base Qwen 1.5B,What are the distinct mechanisms of refractory...,Synergistic corrosion of MgO-C and spinel refr...,mechanism + material + process condition + ref...
2,3,Base Qwen 1.5B,What specific operational parameters and lance...,BOF skull formation lance height prevention,mechanism + lance tip design + post-combustion...
3,4,Base Qwen 1.5B,What specific operational parameters and lance...,Basic oxygen furnace high scrap ratio post com...,lance configuration + lance height + secondary...
4,5,Base Qwen 1.5B,What specific operational parameters and lance...,BOF oxygen lance Mach number 2.0 supersonic no...,mechanism + lance configuration + post-combust...
5,1,Real-only LoRA,What are the distinct mechanisms of refractory...,"Influence of decarburized layer thickness, por...","oxidation"" ""spalling"" ""thermal cycling"" ""MgO-C..."
6,2,Real-only LoRA,What are the distinct mechanisms of refractory...,Synergistic corrosion of MgO-C and spinel refr...,Effect of oxygen lance injection on the penetr...
7,3,Real-only LoRA,What specific operational parameters and lance...,BOF skull formation lance height prevention,"secondary oxygen"" ""post-combustion"" lance heig..."
8,4,Real-only LoRA,What specific operational parameters and lance...,Basic oxygen furnace high scrap ratio post com...,"rotating lance tip"" ""post combustion"" BOF"
9,5,Real-only LoRA,What specific operational parameters and lance...,BOF oxygen lance Mach number 2.0 supersonic no...,basic oxygen furnace lance height reduction sk...


## 11. Main conclusions

1. The repository provides a reproducible pipeline for context-aware next-query generation.
2. Validation remains real-only; synthetic examples are used only for training augmentation.
3. Mixed-data training provides a modest improvement in automatic similarity metrics.
4. Blind LLM-as-a-judge evaluation does not show a decisive overall winner.
5. Synthetic augmentation improves novelty and technical specificity, but can reduce groundedness and search quality.
6. The main next priority is more diverse, high-quality research trajectories and stricter synthetic-data validation—not a larger backbone by itself.

## Limitations

- The real-only and mixed experiments were not trained through a perfectly identical final script version.
- The validation set is small and covers a limited number of held-out research questions.
- Some Task 3 source descriptions are synthetic.
- Search-result evaluation is not implemented; the project generates queries for downstream search but does not perform web retrieval.
- External teacher and judge runs depend on API access and may incur cost.

## 12. Reproduction commands

The notebook wraps the following existing entry points:

```powershell
# Synthetic augmentation
$env:TARGET_SYNTHETIC_EXAMPLES="100"
$env:MAX_SOURCE_EXAMPLES="180"
python .\generate_synthetic_train_pilot.py
python .\clean_synthetic_queries.py
python .\analyze_synthetic_dataset.py

# LoRA training
python .\train_lora_pipeline.py `
  --config .\configs\qwen2.5_1.5b_mixed.json

# Deterministic inference
python .\generate_predictions_pipeline.py `
  --config .\configs\qwen2.5_1.5b_mixed_inference.json

# BERTScore
python .\evaluate_bertscore_single.py `
  --predictions .\runs\qwen2.5_1.5b_lora_mixed_predictions.jsonl `
  --output .\results\qwen2.5_1.5b_lora_mixed_bertscore.json

# Blind pairwise LLM judge
python .\evaluate_llm_judge_pairwise.py --repeats 2
```